In [1]:
from dotenv import load_dotenv

In [2]:
from langchain_groq import ChatGroq

In [39]:
llm = ChatGroq(
    model="moonshotai/kimi-k2-instruct-0905",  # ✅ valid
)

In [40]:
llm.invoke("hi")

AIMessage(content='Hey there! How can I help you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 27, 'total_tokens': 38, 'completion_time': 0.022467052, 'completion_tokens_details': None, 'prompt_time': 0.049498497, 'prompt_tokens_details': None, 'queue_time': 0.435800015, 'total_time': 0.071965549}, 'model_name': 'moonshotai/kimi-k2-instruct-0905', 'system_fingerprint': 'fp_241bc7119c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d436b-9383-7641-8469-de4a53e613b8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 27, 'output_tokens': 11, 'total_tokens': 38})

In [13]:
from langchain.tools import tool

In [14]:
@tool
def multiply(a: int, b: int) -> int:
    """
    Multiply two integers.

    Args:
        a (int): The first integer.
        b (int): The second integer.

    Returns:
        int: The product of a and b.
    """
    return a * b

In [17]:
multiply

StructuredTool(name='multiply', description='Multiply two integers.\n\nArgs:\n    a (int): The first integer.\n    b (int): The second integer.\n\nReturns:\n    int: The product of a and b.', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x112e847c0>)

In [18]:
from langchain_core.tools import StructuredTool

In [25]:
from pydantic import BaseModel,Field

In [26]:
class WeatherInput(BaseModel):
    city: str

In [27]:
def get_weather(city: str) -> str:
    """
    Get the weather for a given city.

    Args:
        city (str): The name of the city.

    Returns:
        str: A string describing the weather in the city.
    """
    return f"The weather in {city} is sunny."

In [28]:
weather_tool = StructuredTool.from_function(
    func=get_weather,
    name="get_weather",
    description="Fetches real-time weather data for a city",
    args_schema=WeatherInput,  
)

In [33]:
from typing import ClassVar,Type

In [34]:
class WeatherInput(BaseModel):
    city: str = Field(..., description="City name")
    units: str = Field("metric", description="metric or imperial")

class GetWeatherTool(StructuredTool):
    name: ClassVar[str] = "get_weather"           
    description: ClassVar[str] = (
        "Fetches weather data for a city"
    )
    args_schema: ClassVar[Type[BaseModel]] = WeatherInput

    def _run(self, city: str, units: str = "metric") -> str:
        return get_weather(city, units)